In [0]:
# Control and click to be taken to dashboard
# https://advana-jupiter.cloud.databricks.mil/editor/notebooks/1320206519828331/dashboards/a0a747c1-2e98-4457-b095-9ad5ae3454c5?o=1122916192266323

In [0]:
# ==========================================
# CREATE CREDENTIAL INPUT WIDGETS
# ==========================================
# Notebook widgets provide a secure way to input credentials without
# embedding them directly in code. They appear at the top of the notebook
# as text input boxes.
#
# BENEFITS:
# - No credentials stored in notebook code
# - Easy to update without editing cells
# - Can be parameterized for different users/environments
# ==========================================

# Remove any existing widgets to start fresh with code line below
# dbutils.widgets.removeAll()

# Create three text input widgets:
# 1. API Key (required) - your Gemini API authentication key
# 2. Base URL (pre-filled) - the API endpoint
# 3. Username (optional) - for tracking/logging purposes
#dbutils.widgets.text("api_key", "", "API Key")
dbutils.widgets.text("API Key", "")
dbutils.widgets.text("Base Url", "https://api.genai.mil/v1")
dbutils.widgets.text("Email", "")
dbutils.widgets.text('AI-Prompt',"")
dbutils.widgets.dropdown("Save Generated Dataset",defaultValue='No', choices=['Yes','No'])
dbutils.widgets.text('Save Dataset as :',"")

# Can delete next two lines because of Database metadata helps LLM auto-select table unless want to manually select
#dbutils.widgets.dropdown("Database",defaultValue='DB1', choices=['DB1','DB2','DB3'])
#dbutils.widgets.dropdown("Table",defaultValue='Table1', choices=['Table1','Table2','Table3'])

print("✅ Input widgets created successfully!")
print("")
print("🔑 NEXT STEP: Look at the top of this notebook and fill in:")
print("   1. Your API Key (required)")
print("   2. Verify the Base URL (pre-filled)")
print("   3. Add your username (email)")
print("")
print("Then proceed to Cell 5 to load credentials and connect to the API.")

✅ Input widgets created successfully!

🔑 NEXT STEP: Look at the top of this notebook and fill in:
   1. Your API Key (required)
   2. Verify the Base URL (pre-filled)
   3. Add your username (email)

Then proceed to Cell 5 to load credentials and connect to the API.


In [0]:
# ==========================================
# TECHNICAL NOTE
# ==========================================
# This notebook uses Python's 'requests' library for direct HTTP calls
# rather than the OpenAI Python SDK. This approach:
# - Avoids dependency conflicts (OpenAI SDK requires pydantic v2)
# - Works with standard libraries available on this cluster
# - Provides full transparency into API request/response structure
# ==========================================

import json
import requests

# ==========================================
# LOAD CREDENTIALS FROM WIDGETS
# ==========================================
print("Loading credentials from widgets...")

# Read credentials from notebook widgets (input boxes at the top of the notebook)
api_key = dbutils.widgets.get("API Key")
base_url = dbutils.widgets.get("Base Url")
username = dbutils.widgets.get("Email")

# Validate required fields
if not api_key:
    raise ValueError("❌ API Key is required. Please enter your API key in the 'API Key' widget at the top of the notebook.")

credentials = {
    "api_key": api_key,
    "base_url": base_url,
    "username": username if username else "(not provided)"
}

print(f"✅ Credentials loaded for: {credentials['username']}")

# ==========================================
# INITIALIZE HTTP CLIENT
# ==========================================
base_url = credentials['base_url']
api_key = credentials['api_key']
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# ==========================================
# QUERY AVAILABLE MODELS
# ==========================================
print("\nQuerying available models...")
models_widget = []
try:
    response = requests.get(f"{base_url}/models", headers=headers)
    response.raise_for_status()
    models_data = response.json()
    
    print("\n✅ Available Models:")
    for model in models_data.get('data', []):
        print(f"   • {model.get('id', 'unknown')}")
        models_widget.append(model.get('id', 'unknown'))
    # Can uncomment to update values
    dbutils.widgets.dropdown("AI-Models",defaultValue='gemini-3.7-flash', choices=models_widget)
        
except Exception as e:
    print(f"❌ Failed to retrieve models: {e}")


Loading credentials from widgets...
✅ Credentials loaded for: oscar.g.bailon.ctr@us.navy.mil

Querying available models...

✅ Available Models:
   • gemini-2.5-flash
   • gemini-2.5-pro
   • gemini-3.1-pro-preview
   • gemini-3.5-flash
   • gemini-3.6-flash
   • gemini-3.7-flash
   • gemini-3-flash-preview


In [0]:
# ==========================================
# CREATE SYNTHETIC CARS & SPORTS DATABASE
# ==========================================
# This cell generates synthetic data about cars and sports,
# writes it to Delta tables, and loads it into Databricks.
# ==========================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# ==========================================
# GENERATE CARS DATA
# ==========================================
print("Generating cars dataset...")

# Car attributes
car_brands = ['Toyota', 'Honda', 'Ford', 'Chevrolet', 'BMW', 'Mercedes', 'Audi', 'Tesla', 'Nissan', 'Hyundai']
car_models = ['Sedan', 'SUV', 'Truck', 'Coupe', 'Hatchback', 'Convertible', 'Minivan', 'Crossover']
colors = ['Black', 'White', 'Silver', 'Red', 'Blue', 'Gray', 'Green', 'Yellow', 'Orange']

n_cars = 1000

cars_data = {
    'car_id': range(1, n_cars + 1),
    'brand': [random.choice(car_brands) for _ in range(n_cars)],
    'model': [random.choice(car_models) for _ in range(n_cars)],
    'year': [random.randint(2015, 2024) for _ in range(n_cars)],
    'color': [random.choice(colors) for _ in range(n_cars)],
    'price': [round(random.uniform(15000, 95000), 2) for _ in range(n_cars)],
    'mileage': [random.randint(0, 150000) for _ in range(n_cars)],
    'horsepower': [random.randint(100, 500) for _ in range(n_cars)],
    'fuel_type': [random.choice(['Gasoline', 'Diesel', 'Electric', 'Hybrid']) for _ in range(n_cars)],
    'transmission': [random.choice(['Automatic', 'Manual']) for _ in range(n_cars)]
}

cars_df = pd.DataFrame(cars_data)
print(f"✅ Generated {len(cars_df)} car records")

# ==========================================
# GENERATE SPORTS DATA
# ==========================================
print("\nGenerating sports dataset...")

sports_list = ['Soccer', 'Basketball', 'Baseball', 'Football', 'Tennis', 'Golf', 'Hockey', 'Volleyball', 'Swimming', 'Boxing']
teams = ['Eagles', 'Tigers', 'Lions', 'Sharks', 'Bears', 'Wolves', 'Dragons', 'Hawks', 'Panthers', 'Cobras']

n_sports = 500

sports_data = {
    'event_id': range(1, n_sports + 1),
    'sport_name': [random.choice(sports_list) for _ in range(n_sports)],
    'team_home': [random.choice(teams) for _ in range(n_sports)],
    'team_away': [random.choice(teams) for _ in range(n_sports)],
    'home_score': [random.randint(0, 150) for _ in range(n_sports)],
    'away_score': [random.randint(0, 150) for _ in range(n_sports)],
    'event_date': [datetime.now() - timedelta(days=random.randint(0, 365)) for _ in range(n_sports)],
    'venue': [f"Stadium {random.randint(1, 20)}" for _ in range(n_sports)],
    'attendance': [random.randint(1000, 80000) for _ in range(n_sports)],
    'season': [random.choice(['Spring', 'Summer', 'Fall', 'Winter']) for _ in range(n_sports)]
}

sports_df = pd.DataFrame(sports_data)
print(f"✅ Generated {len(sports_df)} sports event records")

# ==========================================
# GENERATE CAR OWNERS & SPORTS FANS (JOIN TABLE)
# ==========================================
print("\nGenerating car owners and sports fans dataset...")

first_names = ['John', 'Jane', 'Michael', 'Sarah', 'David', 'Emily', 'Robert', 'Lisa', 'James', 'Mary']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Rodriguez', 'Martinez']

n_owners = 800

owners_data = {
    'owner_id': range(1, n_owners + 1),
    'first_name': [random.choice(first_names) for _ in range(n_owners)],
    'last_name': [random.choice(last_names) for _ in range(n_owners)],
    'car_id': [random.randint(1, n_cars) for _ in range(n_owners)],
    'favorite_sport': [random.choice(sports_list) for _ in range(n_owners)],
    'favorite_team': [random.choice(teams) for _ in range(n_owners)],
    'age': [random.randint(18, 75) for _ in range(n_owners)],
    'city': [random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 'Philadelphia', 'San Antonio', 'San Diego', 'Dallas', 'Austin']) for _ in range(n_owners)]
}

owners_df = pd.DataFrame(owners_data)
print(f"✅ Generated {len(owners_df)} car owner / sports fan records")

# ==========================================
# CONVERT TO SPARK DATAFRAMES
# ==========================================
print("\nConverting to Spark DataFrames...")

spark_cars_df = spark.createDataFrame(cars_df)
spark_sports_df = spark.createDataFrame(sports_df)
spark_owners_df = spark.createDataFrame(owners_df)

print("✅ Spark DataFrames created")

# ==========================================
# WRITE TO LOCAL PATH AS PARQUET
# ==========================================
print("\nWriting data to local path...")

# Define local paths
local_base_path = "/tmp/synthetic_data" # DATABASE
path_cars = f"{local_base_path}/cars"
path_sports = f"{local_base_path}/sports_events"
path_owners = f"{local_base_path}/car_owners"

# Write to parquet files
spark_cars_df.write.format("parquet").mode("overwrite").save(path_cars)
print(f"✅ Written {spark_cars_df.count()} records to {path_cars}")

spark_sports_df.write.format("parquet").mode("overwrite").save(path_sports)
print(f"✅ Written {spark_sports_df.count()} records to {path_sports}")

spark_owners_df.write.format("parquet").mode("overwrite").save(path_owners)
print(f"✅ Written {spark_owners_df.count()} records to {path_owners}")

# ==========================================
# READ DATA BACK (WITHOUT SQL)
# ==========================================
print("\n" + "="*50)
print("READING DATA BACK FROM PARQUET")
print("="*50)

# Read the parquet files back into Spark DataFrames
cars_loaded = spark.read.format("parquet").load(path_cars)
sports_events_loaded = spark.read.format("parquet").load(path_sports)
car_owners_loaded = spark.read.format("parquet").load(path_owners)

print("\n✅ Loaded DataFrames from parquet files")

# Verify data counts
print("\n📊 Verifying data:")
print(f"   - cars: {cars_loaded.count()} records")
print(f"   - sports_events: {sports_events_loaded.count()} records")
print(f"   - car_owners: {car_owners_loaded.count()} records")

# Display sample data from each DataFrame
print("\n🔍 Sample data:")
print("\nCars:")
display(cars_loaded.limit(10))

print("\nSports Events:")
display(sports_events_loaded.limit(10))

print("\nCar Owners:")
display(car_owners_loaded.limit(10))

print("\n✅ All synthetic data has been saved and loaded successfully!")

Generating cars dataset...
✅ Generated 1000 car records

Generating sports dataset...
✅ Generated 500 sports event records

Generating car owners and sports fans dataset...
✅ Generated 800 car owner / sports fan records

Converting to Spark DataFrames...
✅ Spark DataFrames created

Writing data to local path...
✅ Written 1000 records to /tmp/synthetic_data/cars
✅ Written 500 records to /tmp/synthetic_data/sports_events
✅ Written 800 records to /tmp/synthetic_data/car_owners

READING DATA BACK FROM PARQUET

✅ Loaded DataFrames from parquet files

📊 Verifying data:
   - cars: 1000 records
   - sports_events: 500 records
   - car_owners: 800 records

🔍 Sample data:

Cars:


car_id,brand,model,year,color,price,mileage,horsepower,fuel_type,transmission
1,Honda,Hatchback,2021,Yellow,78342.66,34019,295,Gasoline,Manual
2,Toyota,Hatchback,2022,Green,92053.7,46829,447,Diesel,Manual
3,BMW,Crossover,2022,White,66003.05,8198,261,Diesel,Automatic
4,Chevrolet,Truck,2018,Red,86208.87,52973,371,Diesel,Automatic
5,Chevrolet,Crossover,2019,Yellow,59986.61,28063,449,Diesel,Manual
6,Ford,Crossover,2020,White,67367.68,127236,121,Gasoline,Automatic
7,Honda,Convertible,2019,Orange,51496.63,50231,149,Hybrid,Manual
8,Nissan,Convertible,2015,Black,42268.34,25887,372,Hybrid,Automatic
9,Honda,Minivan,2016,Gray,81024.88,8340,426,Gasoline,Automatic
10,Hyundai,Crossover,2024,Gray,80327.09,149687,281,Gasoline,Manual



Sports Events:


event_id,sport_name,team_home,team_away,home_score,away_score,event_date,venue,attendance,season
1,Baseball,Eagles,Cobras,129,124,2025-11-06T15:28:05.949816Z,Stadium 3,26451,Winter
2,Football,Tigers,Bears,72,64,2025-10-10T15:28:05.949827Z,Stadium 11,3185,Summer
3,Tennis,Bears,Wolves,68,93,2026-05-08T15:28:05.949831Z,Stadium 2,44603,Winter
4,Hockey,Panthers,Tigers,67,117,2026-04-14T15:28:05.949834Z,Stadium 14,42481,Winter
5,Volleyball,Hawks,Bears,78,3,2026-07-06T15:28:05.949836Z,Stadium 4,23179,Winter
6,Boxing,Eagles,Hawks,106,90,2025-11-04T15:28:05.949839Z,Stadium 17,77919,Fall
7,Basketball,Dragons,Wolves,61,77,2026-02-19T15:28:05.949841Z,Stadium 11,49005,Winter
8,Baseball,Panthers,Bears,1,85,2026-01-08T15:28:05.949843Z,Stadium 7,68677,Summer
9,Basketball,Lions,Tigers,106,132,2025-10-09T15:28:05.949845Z,Stadium 18,2857,Summer
10,Tennis,Hawks,Eagles,8,123,2026-07-10T15:28:05.949848Z,Stadium 2,53844,Winter



Car Owners:


owner_id,first_name,last_name,car_id,favorite_sport,favorite_team,age,city
1,David,Johnson,398,Tennis,Dragons,65,Phoenix
2,Emily,Johnson,164,Volleyball,Hawks,18,New York
3,John,Johnson,855,Golf,Cobras,34,San Antonio
4,Michael,Williams,207,Tennis,Cobras,58,Dallas
5,Sarah,Davis,773,Swimming,Sharks,64,Dallas
6,Michael,Johnson,623,Boxing,Wolves,34,Phoenix
7,Mary,Williams,263,Golf,Cobras,72,Los Angeles
8,John,Rodriguez,343,Tennis,Sharks,59,Houston
9,Michael,Garcia,201,Volleyball,Hawks,29,Houston
10,James,Williams,335,Swimming,Bears,53,Houston



✅ All synthetic data has been saved and loaded successfully!


In [0]:
# ==========================================
# CREATE METADATA DATAFRAME
# ==========================================
# This cell creates a metadata dataframe containing information
# about the three synthetic datasets created earlier.
# ==========================================

import pandas as pd
from datetime import datetime

# Create metadata for each dataset
metadata_data = {
    'dataset_name': ['cars', 'sports_events', 'car_owners'],
    'record_count': [
        cars_loaded.count(),
        sports_events_loaded.count(),
        car_owners_loaded.count()
    ],
    'column_count': [
        len(cars_loaded.columns),
        len(sports_events_loaded.columns),
        len(car_owners_loaded.columns)
    ],
    'columns': [
        ', '.join(cars_loaded.columns),
        ', '.join(sports_events_loaded.columns),
        ', '.join(car_owners_loaded.columns)
    ],
    'storage_path': [
        path_cars,
        path_sports,
        path_owners
    ],
    'format': ['parquet', 'parquet', 'parquet'],
    'created_timestamp': [datetime.now()] * 3,
    'description': [
        'Synthetic car inventory data with brands, models, prices, and specifications',
        'Synthetic sports event data with teams, scores, venues, and attendance',
        'Synthetic car owner data linking people to cars and favorite sports'
    ]
}

# Create pandas dataframe
metadata_df = pd.DataFrame(metadata_data)

# Convert to Spark DataFrame
spark_metadata_df = spark.createDataFrame(metadata_df)

print("✅ Metadata dataframe created successfully!")
print(f"\n📊 Metadata Summary: {len(metadata_df)} datasets documented")

# Display the metadata
print("\n🔍 Dataset Metadata:")
display(spark_metadata_df)

✅ Metadata dataframe created successfully!

📊 Metadata Summary: 3 datasets documented

🔍 Dataset Metadata:


dataset_name,record_count,column_count,columns,storage_path,format,created_timestamp,description
cars,1000,10,"car_id, brand, model, year, color, price, mileage, horsepower, fuel_type, transmission",/tmp/synthetic_data/cars,parquet,2026-09-10T15:28:14.204453Z,"Synthetic car inventory data with brands, models, prices, and specifications"
sports_events,500,10,"event_id, sport_name, team_home, team_away, home_score, away_score, event_date, venue, attendance, season",/tmp/synthetic_data/sports_events,parquet,2026-09-10T15:28:14.204453Z,"Synthetic sports event data with teams, scores, venues, and attendance"
car_owners,800,8,"owner_id, first_name, last_name, car_id, favorite_sport, favorite_team, age, city",/tmp/synthetic_data/car_owners,parquet,2026-09-10T15:28:14.204453Z,Synthetic car owner data linking people to cars and favorite sports


In [0]:
# ==========================================
# CONVERT METADATA TO CSV STRING
# ==========================================
# Convert the metadata dataframe to CSV format
# to provide context for API calls
# ==========================================

import io

# Convert Spark DataFrame to Pandas for easier CSV conversion
metadata_pandas = spark_metadata_df.toPandas()

# Convert to CSV string
csv_buffer = io.StringIO()
metadata_pandas.to_csv(csv_buffer, index=False)
metadata_csv = csv_buffer.getvalue()

print("✅ Metadata converted to CSV format")
print(f"\n📄 CSV Preview (first 500 characters):")
print(metadata_csv[:500])
print("\n💾 Full CSV stored in variable: metadata_csv")

✅ Metadata converted to CSV format

📄 CSV Preview (first 500 characters):
dataset_name,record_count,column_count,columns,storage_path,format,created_timestamp,description
cars,1000,10,"car_id, brand, model, year, color, price, mileage, horsepower, fuel_type, transmission",/tmp/synthetic_data/cars,parquet,2026-09-10 15:28:14.204453,"Synthetic car inventory data with brands, models, prices, and specifications"
sports_events,500,10,"event_id, sport_name, team_home, team_away, home_score, away_score, event_date, venue, attendance, season",/tmp/synthetic_data/sports_events

💾 Full CSV stored in variable: metadata_csv


In [0]:
# ==========================================
# CREATE COMBINED PROMPT FOR SQL GENERATION
# ==========================================
# This cell combines the metadata CSV with the user's AI prompt
# to create a comprehensive prompt that generates SQL code
# based on the available datasets.
# ==========================================

# Get the user's AI prompt from the widget
#user_prompt = dbutils.widgets.get("AI-Prompt")

# Construct the full prompt with metadata context

ai_prompt = dbutils.widgets.get("AI-Prompt")

full_prompt = f"""You are a SQL expert. Below is metadata about available datasets in CSV format:

{metadata_csv}

Based on this metadata, please generate SQL code for the following request:

{ai_prompt}

Include the filepath of the selected table in double asterisks such as : **FilePath** and nothing else on the top line.

Provide only the SQL code without explanations. Use proper filepath references and load with parquet.``  ."""

print("✅ Combined prompt created successfully!")
print("\n" + "="*60)
print("FULL PROMPT PREVIEW:")
print("="*60)
print(full_prompt)
print("="*60)
print(f"\n📊 Metadata CSV length: {len(metadata_csv)} characters")
print(f"📝 User prompt length: {len(ai_prompt)} characters")
print(f"📋 Total prompt length: {len(full_prompt)} characters")

✅ Combined prompt created successfully!

FULL PROMPT PREVIEW:
You are a SQL expert. Below is metadata about available datasets in CSV format:

dataset_name,record_count,column_count,columns,storage_path,format,created_timestamp,description
cars,1000,10,"car_id, brand, model, year, color, price, mileage, horsepower, fuel_type, transmission",/tmp/synthetic_data/cars,parquet,2026-09-10 15:28:14.204453,"Synthetic car inventory data with brands, models, prices, and specifications"
sports_events,500,10,"event_id, sport_name, team_home, team_away, home_score, away_score, event_date, venue, attendance, season",/tmp/synthetic_data/sports_events,parquet,2026-09-10 15:28:14.204453,"Synthetic sports event data with teams, scores, venues, and attendance"
car_owners,800,8,"owner_id, first_name, last_name, car_id, favorite_sport, favorite_team, age, city",/tmp/synthetic_data/car_owners,parquet,2026-09-10 15:28:14.204453,Synthetic car owner data linking people to cars and favorite sports


Based on th

In [0]:
# Define a simple prompt

#print(f"Sending prompt: '{full_prompt}'\n")
ai_model = dbutils.widgets.get("AI-Models")

try:
    # Prepare the request payload
    payload = {
        "model": ai_model,
        "messages": [{"role": "user", "content": full_prompt}],
        "temperature": 0,
        "max_tokens": 500
    }
    
    # Make the API call
    response = requests.post(
        f"{base_url}/chat/completions",
        headers=headers,
        json=payload
    )
    response.raise_for_status()
    response_data = response.json()
    # Extract and render the response
    answer = response_data['choices'][0]['message']['content']
    # make output consistent with a parser that separates table_name and code
    print(answer)

except Exception as e:
    print(f"❌ API call failed: {e}")


**/tmp/synthetic_data/cars**

```sql
SELECT 
    car_id,
    brand,
    model,
    year,
    color,
    price,
    mileage,
    horsepower,
    fuel_type,
    transmission
FROM parquet.`/tmp/synthetic_data/cars`;
```




In [0]:
LLM_Table_AutoSelection = answer.split("**")[1]
# Load data using SQL query from the parquet path
try:

    Preview_table = spark.sql(f"""
            SELECT *
            FROM parquet.`{LLM_Table_AutoSelection}`
            LIMIT 10
        """)
    display(Preview_table)
except:
    Preview_table = spark.sql(f"""
            SELECT 'Table Not Found'
        """)
    display(Preview_table)
    # Can display other tables/column names
    available_tables = [row.storage_path for row in spark_metadata_df.select("storage_path").collect()]
    print(f"❌ Table '{LLM_Table_AutoSelection}' not found. Available tables: {available_tables}")


car_id,brand,model,year,color,price,mileage,horsepower,fuel_type,transmission
1,Honda,Hatchback,2021,Yellow,78342.66,34019,295,Gasoline,Manual
2,Toyota,Hatchback,2022,Green,92053.7,46829,447,Diesel,Manual
3,BMW,Crossover,2022,White,66003.05,8198,261,Diesel,Automatic
4,Chevrolet,Truck,2018,Red,86208.87,52973,371,Diesel,Automatic
5,Chevrolet,Crossover,2019,Yellow,59986.61,28063,449,Diesel,Manual
6,Ford,Crossover,2020,White,67367.68,127236,121,Gasoline,Automatic
7,Honda,Convertible,2019,Orange,51496.63,50231,149,Hybrid,Manual
8,Nissan,Convertible,2015,Black,42268.34,25887,372,Hybrid,Automatic
9,Honda,Minivan,2016,Gray,81024.88,8340,426,Gasoline,Automatic
10,Hyundai,Crossover,2024,Gray,80327.09,149687,281,Gasoline,Manual


In [0]:
# Parse SQL code from answer variable
# Extract only the executable SQL query
try:
    # Split by backticks if markdown formatted
    if '```' in answer:
        parts = answer.split('```')
        # Find the SQL block
        for part in parts:
            if 'sql' in part.lower()[:10]:  # Check if 'sql' appears near the start
                sql_code = part.replace('sql', '', 1).strip()
                break
            elif 'SELECT' in part.upper() or 'WITH' in part.upper():
                sql_code = part.strip()
                break
        else:
            # No code block found, try to extract SQL directly
            sql_code = answer.strip()
    else:
        # No backticks, assume the SQL is in the answer directly
        sql_code = answer.strip()
    
    # Remove any remaining markdown artifacts
    sql_code = sql_code.replace('```sql', '').replace('```', '').strip()
    
    # Store in variable for spark.sql()
    LLM_SQL_output = sql_code
    
    print("✅ SQL code extracted successfully:")
    print("="*60)
    print(LLM_SQL_output)
    print("="*60)
    
except Exception as e:
    print(f"❌ Failed to parse SQL code: {e}")
    LLM_SQL_output = None

✅ SQL code extracted successfully:
SELECT 
    car_id,
    brand,
    model,
    year,
    color,
    price,
    mileage,
    horsepower,
    fuel_type,
    transmission
FROM parquet.`/tmp/synthetic_data/cars`;


In [0]:
for x in ['INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'MERGE', 'GRANT', 'REVOKE']:
    if x in LLM_SQL_output:
        LLM_SQL_output = "Found Banned SQL Function"
        print("Invalid Code Validation Check")
    else:
        print("Safety Validation Check Successful")

Safety Validation Check Successful
Safety Validation Check Successful
Safety Validation Check Successful
Safety Validation Check Successful
Safety Validation Check Successful
Safety Validation Check Successful
Safety Validation Check Successful
Safety Validation Check Successful
Safety Validation Check Successful


In [0]:
try:
    SQL_DF = spark.sql(LLM_SQL_output)
    display(SQL_DF)
    print(f"✅ Query executed successfully. Returned {SQL_DF.count()} rows.")
except Exception as e:
    print(f"❌ SQL execution failed: {e}")
    print(f"\nGenerated SQL was:\n{LLM_SQL_output}")
    SQL_DF = spark.sql(f"""
            SELECT 'SQL Code Error, Please See Code Generated'
        """)
    display(SQL_DF)

car_id,brand,model,year,color,price,mileage,horsepower,fuel_type,transmission
1,Honda,Hatchback,2021,Yellow,78342.66,34019,295,Gasoline,Manual
2,Toyota,Hatchback,2022,Green,92053.7,46829,447,Diesel,Manual
3,BMW,Crossover,2022,White,66003.05,8198,261,Diesel,Automatic
4,Chevrolet,Truck,2018,Red,86208.87,52973,371,Diesel,Automatic
5,Chevrolet,Crossover,2019,Yellow,59986.61,28063,449,Diesel,Manual
6,Ford,Crossover,2020,White,67367.68,127236,121,Gasoline,Automatic
7,Honda,Convertible,2019,Orange,51496.63,50231,149,Hybrid,Manual
8,Nissan,Convertible,2015,Black,42268.34,25887,372,Hybrid,Automatic
9,Honda,Minivan,2016,Gray,81024.88,8340,426,Gasoline,Automatic
10,Hyundai,Crossover,2024,Gray,80327.09,149687,281,Gasoline,Manual


✅ Query executed successfully. Returned 1000 rows.


In [0]:
import io

# Convert Spark DataFrame to Pandas for easier CSV conversion
sql_pandas = SQL_DF.toPandas()

# Convert to CSV string
csv_buffer = io.StringIO()
sql_pandas.to_csv(csv_buffer, index=False)
sql_df_csv = csv_buffer.getvalue()

In [0]:
#sql_df_csv

'car_id,brand,model,year,color,price,mileage,horsepower,fuel_type,transmission\n1,Honda,Hatchback,2021,Yellow,78342.66,34019,295,Gasoline,Manual\n2,Toyota,Hatchback,2022,Green,92053.7,46829,447,Diesel,Manual\n3,BMW,Crossover,2022,White,66003.05,8198,261,Diesel,Automatic\n4,Chevrolet,Truck,2018,Red,86208.87,52973,371,Diesel,Automatic\n5,Chevrolet,Crossover,2019,Yellow,59986.61,28063,449,Diesel,Manual\n6,Ford,Crossover,2020,White,67367.68,127236,121,Gasoline,Automatic\n7,Honda,Convertible,2019,Orange,51496.63,50231,149,Hybrid,Manual\n8,Nissan,Convertible,2015,Black,42268.34,25887,372,Hybrid,Automatic\n9,Honda,Minivan,2016,Gray,81024.88,8340,426,Gasoline,Automatic\n10,Hyundai,Crossover,2024,Gray,80327.09,149687,281,Gasoline,Manual\n11,Audi,Convertible,2018,Silver,82545.4,101868,321,Diesel,Manual\n12,Toyota,Coupe,2023,Green,41557.81,128426,412,Gasoline,Automatic\n13,Toyota,Coupe,2015,Green,76393.31,137840,433,Hybrid,Automatic\n14,Honda,Minivan,2017,Gray,62996.09,67092,239,Electric,Automati

In [0]:
QA_AI_Prompt = """
You are a Quality Assurance Engineer reviewing the output of the LLM, data source and prompt for alignment and accuracy.

Here is the prompt: """ + ai_prompt + """  + with the data source: """ + LLM_Table_AutoSelection + """ from this metadata response """ + answer + """.
Here is the LLM output: """ + sql_df_csv + """.
"""

In [0]:
# Define a simple prompt

try:
    # Prepare the request payload
    payload = {
        "model": ai_model,
        "messages": [{"role": "user", "content": QA_AI_Prompt}],
        "temperature": 0.5,
        "max_tokens": 500
    }
    
    # Make the API call
    response = requests.post(
        f"{base_url}/chat/completions",
        headers=headers,
        json=payload
    )
    response.raise_for_status()
    response_data = response.json()
    # Extract and render the response
    QA_answer = response_data['choices'][0]['message']['content']
    # make output consistent with a parser that separates table_name and code
    print(QA_answer)

except Exception as e:
    print(f"❌ API call failed: {e}")


# 📋 QA Review Report: LLM Output & Data Alignment

---

### 📌 Executive Summary

| Review Parameter | Status | Details |
|---|---|---|
| **Overall Verdict** | **PASSED** | The LLM output aligns directly with the prompt query, schema, and expected data source format. |
| **Schema Match** | **100% Match** | All 10 requested columns from the `SELECT` statement are present in the exact order. |
| **Record Count** | **1,000 Rows** | Complete dataset containing records `1` through `1000`. |
| **Data Integrity** | **Valid** | Data types, field domains, and value ranges conform to synthetic automotive schemas. |

---

### 🔍 Detailed Findings & Evaluation

| Evaluation Category | Finding | QA Assessment |
|---|---|---|
| **Column Projection** | Expected: `car_id`, `brand`, `model`, `year`, `color`, `price`, `mileage`, `horsepower`, `fuel_type`, `transmission`.<br>Delivered: Exact match in header and row elements. | **Aligned** |
| **Data Partitioning & Ordering** | Rows are returned in 4 chunks

In [0]:
# Convert QA_answer to Spark DataFrame
QA_RESPONSE_DF = spark.createDataFrame([{'AI_Feedback': QA_answer}])

# Display the DataFrame
display(QA_RESPONSE_DF)

AI_Feedback
"# 📋 QA Review Report: LLM Output & Data Alignment --- ### 📌 Executive Summary | Review Parameter | Status | Details | |---|---|---| | **Overall Verdict** | **PASSED** | The LLM output aligns directly with the prompt query, schema, and expected data source format. | | **Schema Match** | **100% Match** | All 10 requested columns from the `SELECT` statement are present in the exact order. | | **Record Count** | **1,000 Rows** | Complete dataset containing records `1` through `1000`. | | **Data Integrity** | **Valid** | Data types, field domains, and value ranges conform to synthetic automotive schemas. | --- ### 🔍 Detailed Findings & Evaluation | Evaluation Category | Finding | QA Assessment | |---|---|---| | **Column Projection** | Expected: `car_id`, `brand`, `model`, `year`, `color`, `price`, `mileage`, `horsepower`, `fuel_type`, `transmission`.Delivered: Exact match in header and row elements. | **Aligned** | | **Data Partitioning & Ordering** | Rows are returned in 4 chunks: `1–250`, `751–1000`, `501–750`, and `251–500`. | **Expected behavior**: The SQL query does not specify an `ORDER BY` clause, reflecting standard multi-part Parquet file read order. | | **Value Domain Verification** | • **Year**: 2015–2024• **Price**: Valid 2-decimal floats• **Fuel Type**: Gasoline, Diesel, Hybrid, Electric• **Transmission**: Manual, Automatic | **Consistent & Valid** | | **Format & Delimiters** | Standard comma-separated values (CSV) format with continuous row records and no truncated lines. | **Valid** | --- ### 💡 Recommendations & Observations * **Deterministic Ordering:** If strict sequential ordering (`1` to `1000`) is required downstream, append `ORDER BY car_id ASC` to the SQL query to avoid partition-order output. * **Output Encapsulation:** When passing large CSV payloads to downstream systems, consider specifying code block delimiters or structured formats (e.g., JSON/Parquet) if automated ingestion is intended."


In [0]:
# Save Generated Data
if dbutils.widgets.get("Save Generated Dataset").strip() == 'Yes':
    # Get dataset name from widget
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    dataset_name = dbutils.widgets.get("Save Dataset as :").strip()
    if not dataset_name:
        dataset_name = f"generated_sql_output_{timestamp}"
    else:
        dataset_name = f"{dataset_name}_{timestamp}"
    
    # Define output path
    output_path = f"/tmp/{dataset_name}"
    
    # Check if dataset already exists
    try:
        existing_files = dbutils.fs.ls(output_path)
        print(f"❌ Dataset '{dataset_name}' already exists at {output_path}")
        print(f"⚠️ Please choose a different name or delete the existing dataset first.")
    except Exception:
        # Path doesn't exist, safe to save
        # Save the SQL_DF DataFrame
        SQL_DF.write.format("parquet").mode("overwrite").save(output_path)
        
        print(f"✅ SQL_DF saved successfully to {output_path}")
        print(f"📊 Saved {SQL_DF.count()} rows")